# Approach A: Baseline Classifier

**Typical baseline:** XGBoost trained on simple aggregate statistics of raw consumption data. No temporal awareness, no seasonality handling.

**Features:** mean, std, max, min, range, median, skew, kurtosis, zero_days, pct_above_2std

**Expected result:** Catches theft but generates many false positives — can't distinguish seasonal variation from actual anomalies.

**Infrastructure:** XGBoost classifier trained on Amazon SageMaker AI using SDK v3 (`ModelTrainer`).

In [ ]:
import warnings
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, precision_recall_fscore_support

from utils import (
    get_sagemaker_session, load_sgcc_dataset, preprocess,
    compute_baseline_features, upload_to_s3,
    train_xgboost_on_sagemaker, download_and_load_model, save_results,
    ROLE, PREFIX,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
print("Imports OK")

In [ ]:
# SageMaker session
sagemaker_session, region, bucket = get_sagemaker_session()
s3_client = boto3.client("s3")
print(f"Region: {region} | Bucket: {bucket}")

In [ ]:
# Load and preprocess SGCC dataset
raw_df = load_sgcc_dataset()
customer_ids, labels, consumption_clean, date_columns, train_idx, test_idx = preprocess(raw_df)

## Exploratory Data Analysis

In [ ]:
# Compare normal vs theft consumption patterns
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
np.random.seed(42)
normal_idx = np.random.choice(labels[labels == 0].index, 5, replace=False)
theft_idx = np.random.choice(labels[labels == 1].index, 5, replace=False)

ax = axes[0, 0]
for idx in normal_idx:
    ax.plot(date_columns, consumption_clean.iloc[idx].values, alpha=0.6, linewidth=0.5)
ax.set_title("Normal Customers (sample of 5)")
ax.set_ylabel("Daily kWh")

ax = axes[0, 1]
for idx in theft_idx:
    ax.plot(date_columns, consumption_clean.iloc[idx].values, alpha=0.6, linewidth=0.5, color="red")
ax.set_title("Theft Customers (sample of 5)")
ax.set_ylabel("Daily kWh")

ax = axes[1, 0]
mean_cons = consumption_clean.mean(axis=1)
ax.hist(mean_cons[labels == 0], bins=50, alpha=0.7, label="Normal", density=True)
ax.hist(mean_cons[labels == 1], bins=50, alpha=0.7, label="Theft", color="red", density=True)
ax.set_title("Distribution of Mean Daily Consumption")
ax.set_xlabel("Mean kWh/day")
ax.set_xlim(0, 50)
ax.legend()

ax = axes[1, 1]
missing_rate = consumption_clean.eq(0).mean(axis=1)
ax.hist(missing_rate[labels == 0], bins=50, alpha=0.7, label="Normal", density=True)
ax.hist(missing_rate[labels == 1], bins=50, alpha=0.7, label="Theft", color="red", density=True)
ax.set_title("Zero-Consumption Rate per Customer")
ax.set_xlabel("Fraction of zero days")
ax.legend()

plt.suptitle("SGCC Dataset — Exploratory Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Feature Engineering & Training

In [ ]:
# Compute baseline features
print("Computing baseline features...")
features_a = compute_baseline_features(consumption_clean)
features_a["FLAG"] = labels.values
print(f"Feature set: {features_a.shape[1] - 1} features")
features_a.describe()

In [ ]:
# Split and upload to S3
train_a = features_a.iloc[train_idx]
test_a = features_a.iloc[test_idx]

train_s3 = upload_to_s3(train_a, f"{PREFIX}/approach-a/data/train/train.csv", bucket, s3_client)
test_s3 = upload_to_s3(test_a, f"{PREFIX}/approach-a/data/test/test.csv", bucket, s3_client)

In [ ]:
# Train on SageMaker
trainer_a = train_xgboost_on_sagemaker(
    "approach-a", train_s3, test_s3,
    region=region, bucket=bucket, sagemaker_session=sagemaker_session,
)

In [ ]:
# Load model and evaluate
model_a, metrics_a = download_and_load_model(trainer_a, s3_client)
X_test = test_a.drop(columns=["FLAG"])
y_test = test_a["FLAG"]

y_pred = model_a.predict(X_test)
y_prob = model_a.predict_proba(X_test)[:, 1]

print("\n" + "=" * 60)
print("APPROACH A: BASELINE CLASSIFIER")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Normal", "Theft"]))

In [ ]:
# Save results for comparison notebook
save_results("approach-a", y_test, y_pred, y_prob, feature_names=list(X_test.columns))

## Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
importances = model_a.feature_importances_
feature_names = list(X_test.columns)
sorted_idx = np.argsort(importances)
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color="#3498db")
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel("Feature Importance")
ax.set_title("Approach A: Baseline Feature Importance")
plt.tight_layout()
plt.show()

## Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "Theft"], yticklabels=["Normal", "Theft"], ax=ax)
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
ax.set_title("Approach A: Confusion Matrix")
plt.tight_layout()
plt.show()

print(f"False Positives: {cm[0,1]} (wasted inspections)")
print(f"True Positives:  {cm[1,1]} (caught theft)")
print(f"False Negatives: {cm[1,0]} (missed theft)")